In [173]:
import numpy as np
import pandas as pd


In [174]:
df=pd.read_csv("loan_data.csv")
df.head()

,person_age,person_gender,person_education,person_income,person_emp_exp,person_home_ownership,loan_amnt,loan_intent,loan_int_rate,loan_percent_income,cb_person_cred_hist_length,credit_score,previous_loan_defaults_on_file,loan_status
0,22.0,female,Master,71948.0,0,RENT,35000.0,PERSONAL,16.02,0.49,3.0,561,No,1
1,21.0,female,High School,12282.0,0,OWN,1000.0,EDUCATION,11.14,0.08,2.0,504,Yes,0
2,25.0,female,High School,12438.0,3,MORTGAGE,5500.0,MEDICAL,12.87,0.44,3.0,635,No,1
3,23.0,female,Bachelor,79753.0,0,RENT,35000.0,MEDICAL,15.23,0.44,2.0,675,No,1
4,24.0,male,Master,66135.0,1,RENT,35000.0,MEDICAL,14.27,0.53,4.0,586,No,1


In [175]:
print(df['loan_intent'].unique())

<StringArray>
[         'PERSONAL',         'EDUCATION',           'MEDICAL',
           'VENTURE',   'HOMEIMPROVEMENT', 'DEBTCONSOLIDATION']
Length: 6, dtype: str


In [176]:
df=df.copy()
df = pd.get_dummies(df, columns=['person_gender'], drop_first=True)
df = pd.get_dummies(df, columns=['loan_intent'], drop_first=True)

df['person_education'] = df['person_education'].map({
    'High School': 1,
    'Associate': 2,
    'Bachelor': 3,
    'Master': 4,
    'Doctorate': 5
})

df['person_home_ownership'] = df['person_home_ownership'].map({
    'RENT': 1,
    'OWN': 2,
    'MORTGAGE': 3,
    'OTHER': 4
})

df['previous_loan_defaults_on_file'] = df['previous_loan_defaults_on_file'].map({
    'No': False,
    'Yes': True
})

print(len(df.columns))

18


In [177]:
col=df.corr()['loan_status']

relation=col[abs(col)>0.1].index.tolist()

df=df[relation]

print(len(df.columns))


7


In [178]:
print(len(df))

45000


In [179]:
numeric_cols = df.select_dtypes(include=['float64', 'int64']).columns

Q1 = df[numeric_cols].quantile(0.25)
Q3 = df[numeric_cols].quantile(0.75)

IQR = Q3 - Q1

upper = Q3 + 1.5 * IQR
lower = Q1 - 1.5 * IQR

df = df[~((df[numeric_cols] > upper) | (df[numeric_cols] < lower)).any(axis=1)]


In [ ]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import cross_val_score
from sklearn.preprocessing import StandardScaler
import numpy as np

scaler = StandardScaler()
X = df.drop(columns=['loan_status'])
X = scaler.fit_transform(X)
Y = df['loan_status']

limit = int(np.sqrt(len(df)))
sol = []

for k in range(1, limit,2):
    model = KNeighborsClassifier(n_neighbors=k)
    scores = cross_val_score(model, X, Y, cv=5)
    sol.append({'k': k, 'score': scores.mean()})

best = max(sol, key=lambda x: x['score'])
print(f"Best accuracy: {best['score']:.4f}")
print(f"Best K value: {best['k']}")